# 02. Transform

**Цель ноутбука:** применить решения, сформулированные по итогам eda, и получить готовый к агрегации DataFrame.

**План:**
1. Удалить полные дубли строк
2. Добавить флаг `is_price_missing` для строк с `price = 0`
3. Распарсить `category_code` на уровни
4. Добавить производные колонки: event_date, event_hour, day_of_week
5. Сохранить результат для использования на следующем этапе

## 0. Импорты и инициализация

In [1]:
import sys
sys.path.append("..")

from pyspark.sql.functions import (
    col, when, split, to_date, hour, dayofweek
)

from utils.spark_session import get_spark_session
from utils.config import RAW_DATA_PATH, PROCESSED_DATA_DIR
from utils.schema import ec_schema

spark = get_spark_session()

In [2]:
df = (
    spark.read
    .option("header", True)
    .schema(ec_schema)
    .csv(RAW_DATA_PATH)
)

df.show(5, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-10-01 00:00:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 00:00:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-10-01 00:00:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-10-01 00:0

## 1. Удаление полных дублей
Дубли - полностью повторяющиеся строки, которые скорее я вляются ошибкой логирования, чем отдельными событиямия.

In [3]:
before = df.count()
df = df.dropDuplicates()
after = df.count()

print(f"Было строк: {before:,}")
print(f"Стало строк: {after:,}")
print(f"Удалено дублей: {before - after:,}")

Было строк: 42,448,764
Стало строк: 42,418,544
Удалено дублей: 30,220


## 2. Флаг is_price_missing
Строки с `price = 0` валидны для анализа воронки,
но стоит добавить флаг для исключения из денежных метрик.

In [4]:
df = df.withColumn("is_price_missing", when(col("price") == 0, True).otherwise(False))

df.groupBy("is_price_missing").count().show()

+----------------+--------+
|is_price_missing|   count|
+----------------+--------+
|            true|   68670|
|           false|42349874|
+----------------+--------+



## 3. Разделение category_code на уровни

`category_code` — это путь вида `electronics.smartphone`. Разбиваем на отдельные колонки.

Пропуски в `category_code` (32%) оставляем.

In [5]:
category_parts = split(col("category_code"), "\\.")

df = (
    df
    .withColumn("category_l1", category_parts.getItem(0))
    .withColumn("category_l2", category_parts.getItem(1))
    .withColumn("category_l3", category_parts.getItem(2))
    .withColumn("category_l4", category_parts.getItem(3))
)

df.select("category_code", "category_l1", "category_l2", "category_l3", "category_l4").distinct().show(5, truncate=False)

+-----------------------------------+-----------+-----------+-----------+-----------+
|category_code                      |category_l1|category_l2|category_l3|category_l4|
+-----------------------------------+-----------+-----------+-----------+-----------+
|electronics.audio.music_tools.piano|electronics|audio      |music_tools|piano      |
|apparel.shoes.keds                 |apparel    |shoes      |keds       |NULL       |
|apparel.sock                       |apparel    |sock       |NULL       |NULL       |
|appliances.kitchen.kettle          |appliances |kitchen    |kettle     |NULL       |
|computers.components.motherboard   |computers  |components |motherboard|NULL       |
+-----------------------------------+-----------+-----------+-----------+-----------+
only showing top 5 rows



## 4. Производные колонки даты и времени

`event_date`, `event_hour`, `day_of_week` посчитаем сейчас один раз, чтобы не делать это потом в каждом агрегате отдельно.

In [6]:
df = (
    df
    .withColumn("event_date", to_date(col("event_time")))
    .withColumn("event_hour", hour(col("event_time")))
    .withColumn("day_of_week", ((dayofweek(col("event_time")) + 5) % 7) + 1)
)

df.dropDuplicates(["day_of_week"]).select("event_time", "event_date", "event_hour", "day_of_week").show(7, truncate=False)

+-------------------+----------+----------+-----------+
|event_time         |event_date|event_hour|day_of_week|
+-------------------+----------+----------+-----------+
|2019-10-01 00:00:00|2019-10-01|0         |2          |
|2019-10-02 00:00:00|2019-10-02|0         |3          |
|2019-10-03 00:00:00|2019-10-03|0         |4          |
|2019-10-04 00:00:00|2019-10-04|0         |5          |
|2019-10-05 00:00:00|2019-10-05|0         |6          |
|2019-10-06 00:00:00|2019-10-06|0         |7          |
|2019-10-07 00:00:01|2019-10-07|0         |1          |
+-------------------+----------+----------+-----------+



## Проверка итоговой схемы

In [7]:
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)
 |-- is_price_missing: boolean (nullable = false)
 |-- category_l1: string (nullable = true)
 |-- category_l2: string (nullable = true)
 |-- category_l3: string (nullable = true)
 |-- category_l4: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)



## Сохранение результата

Записываем в Parquet — компактный колоночный формат, читается Spark значительно быстрее,
чем CSV, и сохраняет схему (типы данных не нужно указывать заново при чтении).

Партиционируем по `event_date` — 31 уникальное значение. Большинство будущих витрин будут фильтроваться/группироваться по дате.

In [8]:
output_path = f"{PROCESSED_DATA_DIR}/events_clean.parquet"

df.write.mode("overwrite").partitionBy("event_date").parquet(output_path)